<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/ESG%20stock%20capital%20Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# esg_capital_stock_standalone.py
"""
Standalone program for ESG-Tobin's Q analysis using ESG Capital Stock models.
Creates ESG capital stock with DIFFERENT depreciation rates for E, S, and G pillars.
Tests: E_capital(t-1), S_capital(t-1), G_capital(t-1) → Tobin's Q(t) with controls at t
"""
from google.colab import drive
drive.mount('/content/drive')

!pip install -q linearmodels pandas numpy matplotlib seaborn statsmodels openpyxl xlsxwriter

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

class ESGCapitalStockAnalysis:
    """
    Enhanced class for running ESG Capital Stock models on Tobin's Q.
    Creates ESG capital stock with DIFFERENT depreciation rates for E, S, and G.
    """

    def __init__(self, data_path, output_dir):
        """
        Initialize the ESG Capital Stock analysis.
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.df = None
        self.capital_stock_data = {}
        self.results = {}
        self.regression_df = None
        self.best_rates = {'E': None, 'S': None, 'G': None}
        self.comparison_tables = {'E': None, 'S': None, 'G': None}
        self.final_results_table = None
        self.final_detailed_results = None

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Setup plotting style
        plt.style.use('seaborn-v0_8-whitegrid')
        sns.set_palette("husl")

    def load_and_prepare_data(self):
        """
        Load data and prepare basic transformations.
        """
        print("=" * 70)
        print("LOADING AND PREPARING DATA FOR ESG CAPITAL STOCK ANALYSIS")
        print("=" * 70)

        try:
            # Load data
            self.df = pd.read_excel(self.data_path)
            print(f"✓ Loaded data from: {self.data_path}")

            # Set multi-index
            if 'Firm' in self.df.columns and 'Year' in self.df.columns:
                # Ensure Year is numeric
                self.df['Year'] = pd.to_numeric(self.df['Year'], errors='coerce')
                self.df = self.df.set_index(['Firm', 'Year']).sort_index()
                print("✓ Set multi-index (Firm, Year)")
            else:
                print("⚠ Warning: 'Firm' and/or 'Year' columns not found")
                # Try to use first two columns as index
                first_col = self.df.columns[0]
                second_col = self.df.columns[1]
                self.df = self.df.set_index([first_col, second_col]).sort_index()
                print(f"  Using {self.df.index.names} as index")

            # Rename columns if needed
            column_renames = {}
            if "Tobin's Q" in self.df.columns:
                column_renames["Tobin's Q"] = "Tobin_Q"

            # Apply renames if any
            if column_renames:
                self.df = self.df.rename(columns=column_renames)
                print(f"✓ Renamed columns: {column_renames}")

            # Create basic control variables (these will be used at time t)
            if 'Total Assets' in self.df.columns:
                self.df['Size'] = np.log(self.df['Total Assets'].replace(0, np.nan))
                print("✓ Created Size variable (log of Total Assets)")

            if 'Total Liabilities' in self.df.columns and 'Total Assets' in self.df.columns:
                self.df['Leverage'] = self.df['Total Liabilities'] / self.df['Total Assets'].replace(0, np.nan)
                print("✓ Created Leverage variable")

            # ROA should already exist, but if not, create it
            if 'ROA' not in self.df.columns and 'Net Income' in self.df.columns and 'Total Assets' in self.df.columns:
                self.df['ROA'] = self.df['Net Income'] / self.df['Total Assets'].replace(0, np.nan)
                print("✓ Created ROA variable")

            # Create log transformation of Tobin's Q
            if 'Tobin_Q' in self.df.columns:
                self.df['Tobin_Q_log'] = np.log(self.df['Tobin_Q'].replace(0, np.nan) + 0.001)
                print("✓ Created Tobin_Q_log variable")

            print(f"\n✓ Loaded {len(self.df)} observations")
            print(f"✓ Number of firms: {self.df.index.get_level_values(0).nunique()}")

            # Safely get year range
            years = self.df.index.get_level_values(1)
            years_numeric = pd.to_numeric(years, errors='coerce').dropna()
            if len(years_numeric) > 0:
                print(f"✓ Years: {int(years_numeric.min())} to {int(years_numeric.max())}")

            return self.df

        except Exception as e:
            print(f"✗ Error loading data: {e}")
            import traceback
            traceback.print_exc()
            return None

    def create_sector_variable(self):
        """
        Create Consumer Staples sector variable.
        """
        print("\n" + "-" * 50)
        print("CREATING SECTOR VARIABLES")
        print("-" * 50)

        # Define Consumer Staples firms
        consumer_staples_firms = [
            'Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG',
            'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever', 'Haleon Plc'
        ]

        # Get unique firm names
        firm_names = self.df.index.get_level_values(0).unique()
        print(f"Found {len(firm_names)} unique firms")

        # Check which Consumer Staples firms are in the data
        found_cs_firms = [firm for firm in consumer_staples_firms if firm in firm_names]
        print(f"Found {len(found_cs_firms)} Consumer Staples firms in data: {found_cs_firms}")

        # Create sector variable
        consumer_staples_dummy = []
        for firm in self.df.index.get_level_values(0):
            consumer_staples_dummy.append(1 if firm in consumer_staples_firms else 0)

        self.df['ConsumerStaples'] = consumer_staples_dummy

        # Count Consumer Staples observations
        cs_obs = self.df[self.df['ConsumerStaples']==1].shape[0]
        cs_firms = self.df[self.df['ConsumerStaples']==1].index.get_level_values(0).nunique()

        print(f"\n✓ Created sector variables:")
        print(f"  Consumer Staples firms: {cs_firms}")
        print(f"  Consumer Staples observations (raw): {cs_obs}")

        return self.df

    def create_esg_capital_stock(self, e_rate=0.3, s_rate=0.3, g_rate=0.3):
        """
        Create ESG capital stock variables with DIFFERENT depreciation rates for each pillar.
        """
        print(f"\n{'='*70}")
        print(f"CREATING ESG CAPITAL STOCK WITH DIFFERENT RATES")
        print(f"E depreciation: δ_E = {e_rate:.2f}")
        print(f"S depreciation: δ_S = {s_rate:.2f}")
        print(f"G depreciation: δ_G = {g_rate:.2f}")
        print(f"{'='*70}")

        # Reset index to work with years
        df_reset = self.df.reset_index()
        df_reset['Year'] = pd.to_numeric(df_reset['Year'], errors='coerce')
        df_reset = df_reset.sort_values(['Firm', 'Year'])

        # Create capital stock for each component with its own rate
        components = [
            ('E', e_rate, f'E_capital_{int(e_rate*100)}'),
            ('S', s_rate, f'S_capital_{int(s_rate*100)}'),
            ('G', g_rate, f'G_capital_{int(g_rate*100)}')
        ]

        for component, rate, col_name in components:
            if component not in self.df.columns:
                print(f"⚠ {component} not found in data, skipping...")
                continue

            print(f"\nProcessing {component} with δ={rate:.2f} to create {col_name}...")

            # Process each firm separately
            for firm in df_reset['Firm'].unique():
                firm_data = df_reset[df_reset['Firm'] == firm].sort_values('Year')
                firm_indices = firm_data.index

                if len(firm_data) > 0:
                    # Get ESG scores for this firm
                    esg_series = firm_data[component].fillna(0).values

                    # Calculate capital stock using perpetual inventory method
                    capital = np.zeros(len(esg_series))
                    if len(esg_series) > 0:
                        capital[0] = esg_series[0]  # Initial capital
                        for t in range(1, len(esg_series)):
                            capital[t] = (1 - rate) * capital[t-1] + esg_series[t]

                    # Add to dataframe
                    for idx, val in zip(firm_indices, capital):
                        self.df.loc[(firm, firm_data.loc[idx, 'Year']), col_name] = val

            # Print summary
            non_null = self.df[col_name].notna().sum()
            mean_val = self.df[col_name].mean()
            print(f"  ✓ Created {col_name}: {non_null} non-null values, mean={mean_val:.4f}")

        return self.df

    def prepare_regression_dataset(self, e_rate=0.3, s_rate=0.3, g_rate=0.3):
        """
        Prepare dataset with different depreciation rates for each pillar.
        """
        print(f"\n{'='*70}")
        print(f"PREPARING REGRESSION DATASET WITH DIFFERENT RATES")
        print(f"E: δ={e_rate:.2f}, S: δ={s_rate:.2f}, G: δ={g_rate:.2f}")
        print(f"Specification: ESG Capital(t-1) → Tobin's Q(t) with controls at t")
        print(f"{'='*70}")

        # Ensure capital stock exists with these rates
        self.create_esg_capital_stock(e_rate, s_rate, g_rate)

        # Reset index to work with years
        df_reset = self.df.reset_index()
        df_reset['Year'] = pd.to_numeric(df_reset['Year'], errors='coerce')
        df_reset = df_reset.sort_values(['Firm', 'Year'])

        # Get capital stock column names with these specific rates
        capital_cols = {
            'E': f'E_capital_{int(e_rate*100)}',
            'S': f'S_capital_{int(s_rate*100)}',
            'G': f'G_capital_{int(g_rate*100)}'
        }

        # Create regression dataset
        regression_data = []

        for firm in df_reset['Firm'].unique():
            firm_data = df_reset[df_reset['Firm'] == firm].sort_values('Year')

            for i in range(1, len(firm_data)):
                current = firm_data.iloc[i]      # Year t
                previous = firm_data.iloc[i-1]    # Year t-1

                obs = {
                    'Firm': firm,
                    'Year': current['Year'],

                    # Dependent variable (current year t)
                    'Tobin_Q': current.get('Tobin_Q', np.nan),

                    # ESG Capital from t-1 with specific rates
                    'E_capital_lag': previous.get(capital_cols['E'], np.nan),
                    'S_capital_lag': previous.get(capital_cols['S'], np.nan),
                    'G_capital_lag': previous.get(capital_cols['G'], np.nan),

                    # Control variables at time t
                    'Size': current.get('Size', np.nan),
                    'Leverage': current.get('Leverage', np.nan),
                    'ROA': current.get('ROA', np.nan),

                    # Sector indicator
                    'ConsumerStaples': current.get('ConsumerStaples', 0)
                }

                regression_data.append(obs)

        # Create dataframe
        self.regression_df = pd.DataFrame(regression_data)

        if len(self.regression_df) > 0:
            self.regression_df['Year'] = pd.to_numeric(self.regression_df['Year'], errors='coerce')
            self.regression_df = self.regression_df.set_index(['Firm', 'Year']).sort_index()

        # Print summary
        print(f"\nRegression Dataset Summary:")
        print(f"  Total observations: {len(self.regression_df)}")
        print(f"  Firms: {len(self.regression_df.index.get_level_values(0).unique()) if len(self.regression_df) > 0 else 0}")

        if len(self.regression_df) > 0:
            years = self.regression_df.index.get_level_values(1)
            years_numeric = pd.to_numeric(years, errors='coerce').dropna()
            if len(years_numeric) > 0:
                print(f"  Years: {int(years_numeric.min())} to {int(years_numeric.max())}")

        return self.regression_df

    def find_optimal_rate_for_pillar(self, pillar, rates=[0.1, 0.2, 0.3, 0.4, 0.5],
                                    fixed_rates={'E': 0.3, 'S': 0.3, 'G': 0.3}):
        """
        Find optimal depreciation rate for a specific pillar while holding others fixed.
        """
        print(f"\n{'='*70}")
        print(f"FINDING OPTIMAL RATE FOR {pillar} PILLAR")
        print(f"Fixed rates: E={fixed_rates['E']:.2f}, S={fixed_rates['S']:.2f}, G={fixed_rates['G']:.2f}")
        print(f"{'='*70}")

        comparison_data = []

        for rate in rates:
            print(f"\n{'─'*50}")
            print(f"Testing {pillar} δ = {rate:.1f}")
            print(f"{'─'*50}")

            # Set rates: vary the target pillar, keep others fixed
            e_rate = rate if pillar == 'E' else fixed_rates['E']
            s_rate = rate if pillar == 'S' else fixed_rates['S']
            g_rate = rate if pillar == 'G' else fixed_rates['G']

            # Prepare dataset with these rates
            self.prepare_regression_dataset(e_rate, s_rate, g_rate)

            if self.regression_df is None or len(self.regression_df) == 0:
                print(f"  ✗ Failed to prepare dataset")
                continue

            # Run regression
            df_model = self.regression_df.copy()

            # Define variables
            capital_vars = ['E_capital_lag', 'S_capital_lag', 'G_capital_lag']
            control_vars = ['Size', 'Leverage', 'ROA', 'ConsumerStaples']

            all_vars = capital_vars + control_vars + ['Tobin_Q']
            df_clean = df_model.dropna(subset=all_vars)

            if len(df_clean) < 10:
                print(f"  ✗ Insufficient data: {len(df_clean)} observations")
                continue

            # Run regression
            y = df_clean['Tobin_Q']
            X = df_clean[capital_vars + control_vars]
            X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

            try:
                model = PanelOLS(y, X, entity_effects=False, time_effects=False)
                results = model.fit(cov_type='robust')

                # Store results
                row = {
                    f'{pillar}_Rate': rate,
                    'R_Squared': results.rsquared,
                    'Observations': results.nobs,
                    'E_coeff': results.params.get('E_capital_lag', np.nan),
                    'E_pval': results.pvalues.get('E_capital_lag', 1.0),
                    'S_coeff': results.params.get('S_capital_lag', np.nan),
                    'S_pval': results.pvalues.get('S_capital_lag', 1.0),
                    'G_coeff': results.params.get('G_capital_lag', np.nan),
                    'G_pval': results.pvalues.get('G_capital_lag', 1.0)
                }

                # Add significance stars
                for comp in ['E', 'S', 'G']:
                    pval = row[f'{comp}_pval']
                    if pval < 0.01:
                        row[f'{comp}_sig'] = '***'
                    elif pval < 0.05:
                        row[f'{comp}_sig'] = '**'
                    elif pval < 0.10:
                        row[f'{comp}_sig'] = '*'
                    else:
                        row[f'{comp}_sig'] = ''

                comparison_data.append(row)

                print(f"  ✓ R² = {results.rsquared:.4f}, N = {results.nobs}")
                print(f"    {pillar} coeff = {row[f'{pillar}_coeff']:.4f}{row[f'{pillar}_sig']} (p={row[f'{pillar}_pval']:.4f})")

            except Exception as e:
                print(f"  ✗ Regression failed: {e}")

        if comparison_data:
            comp_df = pd.DataFrame(comparison_data)

            # Find optimal rate (highest R-squared)
            best_idx = comp_df['R_Squared'].idxmax()
            best_rate = comp_df.loc[best_idx, f'{pillar}_Rate']

            print(f"\n{'='*70}")
            print(f"OPTIMAL RATE FOR {pillar} PILLAR: δ = {best_rate:.2f}")
            print(f"R-squared at optimal rate: {comp_df.loc[best_idx, 'R_Squared']:.4f}")

            # Store results
            self.comparison_tables[pillar] = comp_df
            self.best_rates[pillar] = best_rate

            return comp_df

        return None

    def find_all_optimal_rates(self, rates=[0.1, 0.2, 0.3, 0.4, 0.5], max_iterations=3):
        """
        Iteratively find optimal rates for all pillars.
        """
        print("\n" + "=" * 70)
        print("FINDING OPTIMAL RATES FOR ALL PILLARS (ITERATIVE)")
        print("=" * 70)

        # Start with default rates
        current_rates = {'E': 0.3, 'S': 0.3, 'G': 0.3}

        for iteration in range(max_iterations):
            print(f"\n{'─'*70}")
            print(f"ITERATION {iteration + 1}")
            print(f"{'─'*70}")
            print(f"Current rates: E={current_rates['E']:.2f}, S={current_rates['S']:.2f}, G={current_rates['G']:.2f}")

            new_rates = current_rates.copy()

            # Optimize each pillar one by one
            for pillar in ['E', 'S', 'G']:
                print(f"\n>>> Optimizing {pillar} pillar...")

                # Find best rate for this pillar given current rates for others
                comp_df = self.find_optimal_rate_for_pillar(
                    pillar=pillar,
                    rates=rates,
                    fixed_rates=current_rates
                )

                if comp_df is not None:
                    best_idx = comp_df['R_Squared'].idxmax()
                    new_rates[pillar] = comp_df.loc[best_idx, f'{pillar}_Rate']
                    print(f"  → New optimal {pillar} rate: {new_rates[pillar]:.2f}")

            # Check if converged
            if (new_rates['E'] == current_rates['E'] and
                new_rates['S'] == current_rates['S'] and
                new_rates['G'] == current_rates['G']):
                print(f"\n✓ Converged after {iteration + 1} iterations!")
                current_rates = new_rates
                break

            current_rates = new_rates

        # Store final optimal rates
        self.best_rates = current_rates

        print(f"\n{'='*70}")
        print("FINAL OPTIMAL RATES:")
        print(f"{'='*70}")
        print(f"E pillar: δ = {self.best_rates['E']:.2f}")
        print(f"S pillar: δ = {self.best_rates['S']:.2f}")
        print(f"G pillar: δ = {self.best_rates['G']:.2f}")

        return self.best_rates

    def run_final_model(self, sector='All'):
        """
        Run final model with optimal rates for each pillar.
        """
        if None in self.best_rates.values():
            print("✗ Optimal rates not found. Run find_all_optimal_rates() first.")
            return None

        print("\n" + "=" * 70)
        print(f"FINAL MODEL WITH OPTIMAL PILLAR-SPECIFIC RATES")
        print(f"E: δ={self.best_rates['E']:.2f}, S: δ={self.best_rates['S']:.2f}, G: δ={self.best_rates['G']:.2f}")
        print(f"Sector: {sector}")
        print("=" * 70)

        # Prepare dataset with optimal rates
        self.prepare_regression_dataset(
            e_rate=self.best_rates['E'],
            s_rate=self.best_rates['S'],
            g_rate=self.best_rates['G']
        )

        if self.regression_df is None or len(self.regression_df) == 0:
            print("✗ Regression dataset is empty")
            return None

        # Filter by sector
        df_model = self.regression_df.copy()
        if sector == 'CS':
            if 'ConsumerStaples' in df_model.columns:
                df_model = df_model[df_model['ConsumerStaples'] == 1]
                print(f"Filtered to Consumer Staples: {len(df_model)} observations")
            else:
                print("⚠ ConsumerStaples not found, using all data")

        # Define variables
        capital_vars = ['E_capital_lag', 'S_capital_lag', 'G_capital_lag']
        control_vars = ['Size', 'Leverage', 'ROA']

        if sector == 'All' and 'ConsumerStaples' in df_model.columns:
            control_vars.append('ConsumerStaples')

        # Check which variables exist
        available_capital = [v for v in capital_vars if v in df_model.columns]
        available_controls = [v for v in control_vars if v in df_model.columns]

        all_vars = available_capital + available_controls + ['Tobin_Q']
        df_clean = df_model.dropna(subset=all_vars)

        if len(df_clean) < 10:
            print(f"\n✗ Insufficient data: Only {len(df_clean)} observations")
            return None

        print(f"\nFinal regression sample:")
        print(f"  Observations: {len(df_clean)}")
        print(f"  Firms: {df_clean.index.get_level_values(0).nunique()}")

        # Prepare regression
        y = df_clean['Tobin_Q']
        X = df_clean[available_capital + available_controls]
        X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

        try:
            # Run regression
            model = PanelOLS(y, X, entity_effects=False, time_effects=False)
            results = model.fit(cov_type='robust')

            # Store results
            sector_name = "All Sectors" if sector == 'All' else "Consumer Staples"
            result_row = {
                'Model': sector_name,
                'E_rate': self.best_rates['E'],
                'S_rate': self.best_rates['S'],
                'G_rate': self.best_rates['G'],
                'const_coeff': results.params.get('const', np.nan),
                'const_pval': results.pvalues.get('const', 1.0),
                'E_coeff': results.params.get('E_capital_lag', np.nan),
                'E_pval': results.pvalues.get('E_capital_lag', 1.0),
                'S_coeff': results.params.get('S_capital_lag', np.nan),
                'S_pval': results.pvalues.get('S_capital_lag', 1.0),
                'G_coeff': results.params.get('G_capital_lag', np.nan),
                'G_pval': results.pvalues.get('G_capital_lag', 1.0),
                'Size_coeff': results.params.get('Size', np.nan),
                'Size_pval': results.pvalues.get('Size', 1.0),
                'Leverage_coeff': results.params.get('Leverage', np.nan),
                'Leverage_pval': results.pvalues.get('Leverage', 1.0),
                'ROA_coeff': results.params.get('ROA', np.nan),
                'ROA_pval': results.pvalues.get('ROA', 1.0),
                'R_Squared': results.rsquared,
                'Observations': results.nobs
            }

            # Initialize or append to final results
            if self.final_detailed_results is None:
                self.final_detailed_results = pd.DataFrame([result_row])
            else:
                self.final_detailed_results = pd.concat([self.final_detailed_results, pd.DataFrame([result_row])], ignore_index=True)

            # Print results
            print(f"\n✓ Regression successful!")
            print(f"  R-squared: {results.rsquared:.4f}")
            print(f"  Observations: {results.nobs}")

            print(f"\n{'='*50}")
            print("REGRESSION RESULTS")
            print(f"{'='*50}")

            print(f"\n{'Variable':<20} {'Coefficient':<15} {'P-value':<15} {'Significance'}")
            print("-" * 65)

            # Constant
            const_stars = "***" if result_row['const_pval'] < 0.01 else "**" if result_row['const_pval'] < 0.05 else "*" if result_row['const_pval'] < 0.10 else ""
            print(f"{'const':<20} {result_row['const_coeff']:>10.4f}        {result_row['const_pval']:>8.4f}        {const_stars}")

            # ESG variables
            for var in ['E', 'S', 'G']:
                stars = "***" if result_row[f'{var}_pval'] < 0.01 else "**" if result_row[f'{var}_pval'] < 0.05 else "*" if result_row[f'{var}_pval'] < 0.10 else ""
                print(f"{var}_capital_lag{'':<8} {result_row[f'{var}_coeff']:>10.4f}        {result_row[f'{var}_pval']:>8.4f}        {stars}")

            # Control variables
            for var in ['Size', 'Leverage', 'ROA']:
                stars = "***" if result_row[f'{var}_pval'] < 0.01 else "**" if result_row[f'{var}_pval'] < 0.05 else "*" if result_row[f'{var}_pval'] < 0.10 else ""
                print(f"{var:<20} {result_row[f'{var}_coeff']:>10.4f}        {result_row[f'{var}_pval']:>8.4f}        {stars}")

            print("-" * 65)
            print(f"{'R-squared':<20} {result_row['R_Squared']:>10.4f}")
            print(f"{'Observations':<20} {result_row['Observations']:>10.0f}")

            return results

        except Exception as e:
            print(f"\n✗ Error running regression: {e}")
            return None

    def save_results_to_excel(self):
        """
        Save all results to Excel with multiple sheets.
        """
        print("\n" + "=" * 70)
        print("SAVING RESULTS TO EXCEL")
        print("=" * 70)

        excel_path = os.path.join(self.output_dir, 'esg_capital_stock_results.xlsx')

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

            # Sheet 1: Overview
            print("Creating Overview sheet...")
            years = self.df.index.get_level_values(1)
            years_numeric = pd.to_numeric(years, errors='coerce').dropna()
            year_range = f"{int(years_numeric.min())} - {int(years_numeric.max())}" if len(years_numeric) > 0 else "N/A"

            overview_data = {
                'Metric': [
                    'Total Observations',
                    'Total Firms',
                    'Years Range',
                    'ESG Components',
                    'Capital Stock Variables Created',
                    'Optimal E Depreciation Rate',
                    'Optimal S Depreciation Rate',
                    'Optimal G Depreciation Rate',
                    'Model Specification'
                ],
                'Value': [
                    len(self.df),
                    self.df.index.get_level_values(0).nunique(),
                    year_range,
                    ', '.join([c for c in ['E', 'S', 'G'] if c in self.df.columns]),
                    len([c for c in self.df.columns if '_capital_' in c]),
                    f"{self.best_rates['E']:.2f}" if self.best_rates['E'] else "Not found",
                    f"{self.best_rates['S']:.2f}" if self.best_rates['S'] else "Not found",
                    f"{self.best_rates['G']:.2f}" if self.best_rates['G'] else "Not found",
                    'Pillar-specific rates: E(t-1), S(t-1), G(t-1) → Tobin\'s Q(t)'
                ]
            }
            pd.DataFrame(overview_data).to_excel(writer, sheet_name='Overview', index=False)

            # Sheet 2-4: Optimization results for each pillar
            for pillar in ['E', 'S', 'G']:
                if self.comparison_tables[pillar] is not None:
                    print(f"Creating {pillar} Optimization sheet...")
                    comp_df = self.comparison_tables[pillar]
                    display_df = pd.DataFrame()
                    display_df[f'{pillar}_Rate'] = comp_df[f'{pillar}_Rate']
                    display_df['R_Squared'] = comp_df['R_Squared'].apply(lambda x: f"{x:.4f}")
                    display_df['Observations'] = comp_df['Observations']

                    for comp in ['E', 'S', 'G']:
                        display_df[f'{comp}_Coef'] = comp_df.apply(
                            lambda row: f"{row[f'{comp}_coeff']:.4f}{row[f'{comp}_sig']}", axis=1)
                        display_df[f'{comp}_pval'] = comp_df[f'{comp}_pval'].apply(lambda x: f"{x:.4f}")

                    display_df.to_excel(writer, sheet_name=f'{pillar}_Optimization', index=False)

            # Sheet 5: Final Results - All Sectors
            if self.final_detailed_results is not None:
                print("Creating Final Results sheets...")

                # All Sectors
                all_sectors = self.final_detailed_results[self.final_detailed_results['Model'] == 'All Sectors']
                if len(all_sectors) > 0:
                    row = all_sectors.iloc[0]
                    all_data = [{
                        'Variable': 'Constant',
                        'Coefficient': f"{row['const_coeff']:.4f}",
                        'P-value': f"{row['const_pval']:.4f}",
                        'Significance': '***' if row['const_pval'] < 0.01 else '**' if row['const_pval'] < 0.05 else '*' if row['const_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'E Capital (t-1) [δ={row["E_rate"]:.2f}]',
                        'Coefficient': f"{row['E_coeff']:.4f}",
                        'P-value': f"{row['E_pval']:.4f}",
                        'Significance': '***' if row['E_pval'] < 0.01 else '**' if row['E_pval'] < 0.05 else '*' if row['E_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'S Capital (t-1) [δ={row["S_rate"]:.2f}]',
                        'Coefficient': f"{row['S_coeff']:.4f}",
                        'P-value': f"{row['S_pval']:.4f}",
                        'Significance': '***' if row['S_pval'] < 0.01 else '**' if row['S_pval'] < 0.05 else '*' if row['S_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'G Capital (t-1) [δ={row["G_rate"]:.2f}]',
                        'Coefficient': f"{row['G_coeff']:.4f}",
                        'P-value': f"{row['G_pval']:.4f}",
                        'Significance': '***' if row['G_pval'] < 0.01 else '**' if row['G_pval'] < 0.05 else '*' if row['G_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'Size (t)',
                        'Coefficient': f"{row['Size_coeff']:.4f}",
                        'P-value': f"{row['Size_pval']:.4f}",
                        'Significance': '***' if row['Size_pval'] < 0.01 else '**' if row['Size_pval'] < 0.05 else '*' if row['Size_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'Leverage (t)',
                        'Coefficient': f"{row['Leverage_coeff']:.4f}",
                        'P-value': f"{row['Leverage_pval']:.4f}",
                        'Significance': '***' if row['Leverage_pval'] < 0.01 else '**' if row['Leverage_pval'] < 0.05 else '*' if row['Leverage_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'ROA (t)',
                        'Coefficient': f"{row['ROA_coeff']:.4f}",
                        'P-value': f"{row['ROA_pval']:.4f}",
                        'Significance': '***' if row['ROA_pval'] < 0.01 else '**' if row['ROA_pval'] < 0.05 else '*' if row['ROA_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'R-squared',
                        'Coefficient': f"{row['R_Squared']:.4f}",
                        'P-value': '',
                        'Significance': ''
                    },
                    {
                        'Variable': 'Observations',
                        'Coefficient': f"{int(row['Observations'])}",
                        'P-value': '',
                        'Significance': ''
                    }]

                    pd.DataFrame(all_data).to_excel(writer, sheet_name='All_Sectors_Results', index=False)

                # Consumer Staples
                cs_sectors = self.final_detailed_results[self.final_detailed_results['Model'] == 'Consumer Staples']
                if len(cs_sectors) > 0:
                    row = cs_sectors.iloc[0]
                    cs_data = [{
                        'Variable': 'Constant',
                        'Coefficient': f"{row['const_coeff']:.4f}",
                        'P-value': f"{row['const_pval']:.4f}",
                        'Significance': '***' if row['const_pval'] < 0.01 else '**' if row['const_pval'] < 0.05 else '*' if row['const_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'E Capital (t-1) [δ={row["E_rate"]:.2f}]',
                        'Coefficient': f"{row['E_coeff']:.4f}",
                        'P-value': f"{row['E_pval']:.4f}",
                        'Significance': '***' if row['E_pval'] < 0.01 else '**' if row['E_pval'] < 0.05 else '*' if row['E_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'S Capital (t-1) [δ={row["S_rate"]:.2f}]',
                        'Coefficient': f"{row['S_coeff']:.4f}",
                        'P-value': f"{row['S_pval']:.4f}",
                        'Significance': '***' if row['S_pval'] < 0.01 else '**' if row['S_pval'] < 0.05 else '*' if row['S_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': f'G Capital (t-1) [δ={row["G_rate"]:.2f}]',
                        'Coefficient': f"{row['G_coeff']:.4f}",
                        'P-value': f"{row['G_pval']:.4f}",
                        'Significance': '***' if row['G_pval'] < 0.01 else '**' if row['G_pval'] < 0.05 else '*' if row['G_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'Size (t)',
                        'Coefficient': f"{row['Size_coeff']:.4f}",
                        'P-value': f"{row['Size_pval']:.4f}",
                        'Significance': '***' if row['Size_pval'] < 0.01 else '**' if row['Size_pval'] < 0.05 else '*' if row['Size_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'Leverage (t)',
                        'Coefficient': f"{row['Leverage_coeff']:.4f}",
                        'P-value': f"{row['Leverage_pval']:.4f}",
                        'Significance': '***' if row['Leverage_pval'] < 0.01 else '**' if row['Leverage_pval'] < 0.05 else '*' if row['Leverage_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'ROA (t)',
                        'Coefficient': f"{row['ROA_coeff']:.4f}",
                        'P-value': f"{row['ROA_pval']:.4f}",
                        'Significance': '***' if row['ROA_pval'] < 0.01 else '**' if row['ROA_pval'] < 0.05 else '*' if row['ROA_pval'] < 0.10 else ''
                    },
                    {
                        'Variable': 'R-squared',
                        'Coefficient': f"{row['R_Squared']:.4f}",
                        'P-value': '',
                        'Significance': ''
                    },
                    {
                        'Variable': 'Observations',
                        'Coefficient': f"{int(row['Observations'])}",
                        'P-value': '',
                        'Significance': ''
                    }]

                    pd.DataFrame(cs_data).to_excel(writer, sheet_name='Consumer_Staples_Results', index=False)

            # Sheet 6: Raw Data Sample
            print("Creating Raw Data Sample sheet...")
            self.df.reset_index().head(100).to_excel(writer, sheet_name='Raw_Data_Sample', index=False)

            print(f"\n✓ Successfully saved all results to: {excel_path}")
            return excel_path


def main():
    """
    Main function to run the analysis with pillar-specific depreciation rates.
    """
    DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx'
    OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/esg_capital_stock_output'

    print("=" * 70)
    print("ESG CAPITAL STOCK ANALYSIS - PILLAR-SPECIFIC RATES")
    print("Finding optimal rates for E, S, and G independently")
    print("=" * 70)

    # Initialize
    analyzer = ESGCapitalStockAnalysis(DATA_PATH, OUTPUT_DIR)

    # Step 1: Load data
    print("\nSTEP 1: LOADING DATA")
    analyzer.load_and_prepare_data()

    # Step 2: Create sector variable
    print("\nSTEP 2: CREATING SECTOR VARIABLES")
    analyzer.create_sector_variable()

    # Step 3: Find optimal rates for all pillars (iterative)
    print("\nSTEP 3: FINDING OPTIMAL PILLAR-SPECIFIC RATES")
    rates_to_test = [0.1, 0.2, 0.3, 0.4, 0.5]  # 10% to 50%
    best_rates = analyzer.find_all_optimal_rates(rates=rates_to_test, max_iterations=3)

    # Step 4: Run final model for All Sectors
    print("\nSTEP 4: FINAL MODEL - ALL SECTORS")
    result_all = analyzer.run_final_model(sector='All')

    # Step 5: Run final model for Consumer Staples
    print("\nSTEP 5: FINAL MODEL - CONSUMER STAPLES")
    result_cs = analyzer.run_final_model(sector='CS')

    # Step 6: Save all results
    print("\nSTEP 6: SAVING RESULTS")
    analyzer.save_results_to_excel()

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)

    # Final summary
    print(f"\nFINAL OPTIMAL RATES:")
    print(f"  Environmental (E): δ = {analyzer.best_rates['E']:.2f}")
    print(f"  Social (S):       δ = {analyzer.best_rates['S']:.2f}")
    print(f"  Governance (G):   δ = {analyzer.best_rates['G']:.2f}")

    print(f"\nResults saved to: {analyzer.output_dir}")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ESG CAPITAL STOCK ANALYSIS - PILLAR-SPECIFIC RATES
Finding optimal rates for E, S, and G independently

STEP 1: LOADING DATA
LOADING AND PREPARING DATA FOR ESG CAPITAL STOCK ANALYSIS
✓ Loaded data from: /content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx
✓ Set multi-index (Firm, Year)
✓ Created Size variable (log of Total Assets)
✓ Created Leverage variable
✓ Created Tobin_Q_log variable

✓ Loaded 388 observations
✓ Number of firms: 40
✓ Years: 2015 to 2024

STEP 2: CREATING SECTOR VARIABLES

--------------------------------------------------
CREATING SECTOR VARIABLES
--------------------------------------------------
Found 40 unique firms
Found 8 Consumer Staples firms in data: ['Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG', 'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever']

✓ Created sector variable